# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emgakii001/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
!git clone https://github.com/emgakii001/flyrank-ml-internship.git

In [2]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship


In [3]:
import os
print("Current folder:", os.getcwd())
print("Files here:", os.listdir("."))

Current folder: /content/flyrank-ml-internship
Files here: ['submission', 'DATA_USE.md', 'index.html', 'notebooks', '.github', 'outputs', 'GUIDE.md', 'requirements.txt', 'CLAUDE.md', 'docs', 'scripts', 'README.md', 'skills', 'data', 'SETUP.md', 'AGENTS.md', '.gitignore', 'LICENSE', 'work', '.git']


# **1. Build the feature vector**

This section builds an expanded feature vector incorporating categorical signals (content_type, position_tier, freshness_tier) alongside the six numeric features used in Weeks 5-6, following the flyrank-data skill's warning that missingness in this dataset follows content_type rather than occurring randomly.

This warning proved accurate: word_count is missing in 7,699 of 30,000 rows (25.7%) — and every single one of those missing rows belongs to the "keyword article" content type, while "comparison article" and "feedly article" pages have complete word count data. A naive fillna(0) would have silently claimed these 7,699 pages have zero words, when the true meaning is "word count was never recorded for this content type" — a materially different fact a model needs to distinguish.

To preserve this distinction, a has_word_count flag was created before filling missing values, recording which rows originally had missing data. This proved necessary in practice: after filling word_count with 0, checking for missing values directly returned zero — the original signal was already gone, and only the pre-saved flag preserved the true missingness pattern. This is a concrete demonstration of why the has_ flag approach matters: filling first and checking later would have permanently erased a real, non-random pattern in the data.

Categorical features (content_type, position_tier, freshness_tier) were one-hot encoded rather than label-encoded, since these categories have no natural numeric ordering.

In [4]:
# Section 1 — build the feature vector with proper missingness handling
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["needs_review"] = (df["trend_direction"] == "down").astype(int)

numeric_features = ["impressions_90d", "clicks_90d", "ctr", "avg_position",
                     "engagement_rate", "content_age_days", "word_count"]

# Missingness flags BEFORE filling — this is what preserved the real signal
for col in numeric_features:
    df[f"has_{col}"] = df[col].notna().astype(int)

for col in numeric_features:
    df[col] = df[col].fillna(0)

categorical_features = ["content_type", "position_tier", "freshness_tier"]
df_encoded = pd.get_dummies(df, columns=categorical_features, prefix=categorical_features)

engineered_features = numeric_features + [f"has_{col}" for col in numeric_features]
categorical_encoded_cols = [c for c in df_encoded.columns if c.startswith(tuple(categorical_features))]
feature_vector_cols = engineered_features + categorical_encoded_cols

feature_vector = df_encoded[["content_id", "client_id"] + feature_vector_cols + ["needs_review"]]

print(f"Feature vector shape: {feature_vector.shape}")
print(f"\nWord count missingness by content type (using preserved flag):")
print(df[df["has_word_count"] == 0]["content_type"].value_counts())

Feature vector shape: (30000, 29)

Word count missingness by content type (using preserved flag):
content_type
keyword article    7699
Name: count, dtype: int64


## **2. Feature notes (meaning, missing, categorical, available-when?)**

The table below documents each feature in the vector: its meaning, how missing values are handled, and whether it is genuinely knowable before the review decision is made — the same "knowable at the decision moment" test applied throughout this project.

All numeric features are trailing 90-day aggregates computed from historical search and engagement data — by definition, these are available before any review decision, since they summarize what has already happened up to that point. The one exception requiring care is word_count, where missingness is not random: it is missing specifically for "keyword article" pages (7,699 of 30,000 rows), handled via a has_word_count flag rather than a blind fill, so the model can distinguish "no content" from "content type doesn't record this field."

The three categorical features (content_type, position_tier, freshness_tier) are also available before the decision — they describe the page's existing state and classification, not any future outcome. None of these features overlap with the label-derived fields (trend_direction, trend_pct) excluded throughout this project.

In [5]:
# Section 2 — feature documentation table

feature_notes = pd.DataFrame([
    {"feature": "impressions_90d", "type": "numeric", "meaning": "Total search impressions, trailing 90 days",
     "missing_handling": "None missing", "available_before_decision": True},
    {"feature": "clicks_90d", "type": "numeric", "meaning": "Total search clicks, trailing 90 days",
     "missing_handling": "None missing", "available_before_decision": True},
    {"feature": "ctr", "type": "numeric", "meaning": "Click-through rate (clicks/impressions)",
     "missing_handling": "None missing", "available_before_decision": True},
    {"feature": "avg_position", "type": "numeric", "meaning": "Average search ranking position",
     "missing_handling": "0 = no data (1,205 rows dataset-wide); not filled further here",
     "available_before_decision": True},
    {"feature": "engagement_rate", "type": "numeric", "meaning": "Share of visits that were engaged",
     "missing_handling": "None missing", "available_before_decision": True},
    {"feature": "content_age_days", "type": "numeric", "meaning": "Days since content was created",
     "missing_handling": "None missing", "available_before_decision": True},
    {"feature": "word_count", "type": "numeric", "meaning": "Word count of the page",
     "missing_handling": "Missing for 'keyword article' pages (25.7%); has_word_count flag preserves this + fillna(0)",
     "available_before_decision": True},
    {"feature": "content_type", "type": "categorical (one-hot)", "meaning": "Article format classification",
     "missing_handling": "None missing", "available_before_decision": True},
    {"feature": "position_tier", "type": "categorical (one-hot)", "meaning": "Bucketed search ranking tier",
     "missing_handling": "None missing", "available_before_decision": True},
    {"feature": "freshness_tier", "type": "categorical (one-hot)", "meaning": "Bucketed days-since-last-update",
     "missing_handling": "None missing", "available_before_decision": True},
])

print(feature_notes.to_string(index=False))

         feature                  type                                    meaning                                                                            missing_handling  available_before_decision
 impressions_90d               numeric Total search impressions, trailing 90 days                                                                                None missing                       True
      clicks_90d               numeric      Total search clicks, trailing 90 days                                                                                None missing                       True
             ctr               numeric    Click-through rate (clicks/impressions)                                                                                None missing                       True
    avg_position               numeric            Average search ranking position                              0 = no data (1,205 rows dataset-wide); not filled further here                       

## 3. The leakage hunt
This section attacks the expanded 26-feature vector built in Sections 1-2, checking for label-derived columns, future-window data, and product-defined flags. None of the six forbidden signals (trend_direction, trend_pct, health_score, priority_score, action_type, is_declining_label) appear in the feature set — confirmed directly rather than assumed.

To verify this isn't just a naming exercise, I repeated the deliberate leakage test used in Weeks 3 and 6: adding trend_pct — the exact field the needs_review label is derived from — as a 27th feature. The honest 26-feature model scored AUC = 0.568. The leaky 27-feature model scored a perfect AUC = 1.000. This confirms two things: the expanded feature set is genuinely clean, and if trend_pct were ever accidentally included, the resulting perfect score would serve as an immediate, unmistakable warning sign.



In [6]:
# Section 3 — the leakage hunt: forbidden-field check + deliberate test

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

forbidden_signals = ["trend_direction", "trend_pct", "health_score",
                      "priority_score", "action_type", "is_declining_label"]

print("Checking feature_vector_cols against forbidden signals:")
for signal in forbidden_signals:
    status = "LEAKED — PROBLEM" if signal in feature_vector_cols else "not used — clean"
    print(f"  {signal}: {status}")

# Deliberate test: add trend_pct back in and measure the score jump
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(feature_vector, groups=feature_vector["client_id"]))

X_train = feature_vector.iloc[train_idx][feature_vector_cols]
y_train = feature_vector.iloc[train_idx]["needs_review"]
X_test = feature_vector.iloc[test_idx][feature_vector_cols]
y_test = feature_vector.iloc[test_idx]["needs_review"]

honest_model = LogisticRegression(max_iter=2000)
honest_model.fit(X_train, y_train)
honest_score = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])

# Add the leaky field
df_leak = df.loc[feature_vector.index, ["trend_pct"]].fillna(0)
X_train_leak = pd.concat([X_train, df_leak.iloc[train_idx]], axis=1)
X_test_leak = pd.concat([X_test, df_leak.iloc[test_idx]], axis=1)

leak_model = LogisticRegression(max_iter=2000)
leak_model.fit(X_train_leak, y_train)
leak_score = roc_auc_score(y_test, leak_model.predict_proba(X_test_leak)[:, 1])

print(f"\nHonest AUC (26 features): {honest_score:.3f}")
print(f"Leaky AUC (+ trend_pct): {leak_score:.3f}")
print(f"Score jump: {honest_score:.3f} -> {leak_score:.3f}")

Checking feature_vector_cols against forbidden signals:
  trend_direction: not used — clean
  trend_pct: not used — clean
  health_score: not used — clean
  priority_score: not used — clean
  action_type: not used — clean
  is_declining_label: not used — clean


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Honest AUC (26 features): 0.568
Leaky AUC (+ trend_pct): 1.000
Score jump: 0.568 -> 1.000


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [7]:
honest_model = LogisticRegression(max_iter=5000)
honest_model.fit(X_train, y_train)
honest_score = roc_auc_score(y_test, honest_model.predict_proba(X_test)[:, 1])
print(f"Honest AUC (higher max_iter): {honest_score:.3f}")

Honest AUC (higher max_iter): 0.567


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 4. What I excluded and why

**Trend direction**:The needs review label is built directly from this field; using it as a feature would let the model see the answer.

**Trend pct**:	Confirmed by the deliberate leakage test in Section 3, where adding it pushed AUC to a perfect 1.000.

**Content id and client id**:	Pseudonymized identifiers, used only for grouping/joining (e.g., the client-grouped split), never as predictive signals:  an ID number carries no genuine information about page performance.

**Avg position raw zeros**:	Not excluded, but handled carefully: avg position = 0 means "no position data," not a literal top rank, so this required awareness rather than blind trust in the raw number.

**Any product-defined score (health_score, priority_score, action_type)**:	Not present in this dataset, but explicitly checked for and confirmed absent,  these would represent a business decision already made, not a raw, learnable signal.


In [8]:
# Section 4 — exclusion summary

excluded_fields = {
    "trend_direction": "Label is derived directly from this field — using it as a feature would leak the answer",
    "trend_pct": "Same leakage risk — confirmed via deliberate test (AUC 0.568 -> 1.000 when included)",
    "content_id": "Pseudonymized ID — used only for row identification, not as a feature",
    "client_id": "Pseudonymized ID — used only for grouped train/test splitting, not as a feature",
    "health_score / priority_score / action_type": "Product-defined scores not present in this dataset; confirmed absent, would represent a pre-made decision rather than raw signal if they existed"
}

print("Fields excluded from the model, and why:")
for field, reason in excluded_fields.items():
    print(f"\n  {field}:")
    print(f"    {reason}")

Fields excluded from the model, and why:

  trend_direction:
    Label is derived directly from this field — using it as a feature would leak the answer

  trend_pct:
    Same leakage risk — confirmed via deliberate test (AUC 0.568 -> 1.000 when included)

  content_id:
    Pseudonymized ID — used only for row identification, not as a feature

  client_id:
    Pseudonymized ID — used only for grouped train/test splitting, not as a feature

  health_score / priority_score / action_type:
    Product-defined scores not present in this dataset; confirmed absent, would represent a pre-made decision rather than raw signal if they existed


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.